In [6]:
# Célula 1: Instalar dependências necessárias no ambiente do Jupyter
# Incluindo Django e suas dependências de DB/REST/GIS
!pip install django djangorestframework django-filter djangorestframework-gis psycopg2-binary
# Incluindo dj_database_url que estava faltando
!pip install dj_database_url
# Incluindo geopy, pandas, faker, requests
!pip install geopy pandas faker requests

# Opcional: Se quiser verificar se foram instaladas (reinicie o kernel após a instalação)
# import pkg_resources
# installed_packages = {d.project_name for d in pkg_resources.working_set}
# print(installed_packages)

In [16]:
# Célula 1: Instalar dependências necessárias no ambiente do Jupyter
# Execute esta célula primeiro. Se instalar algo, REINICIE o kernel (Kernel -> Restart).
!pip install requests pandas faker geopy

# Célula 2: Configurar o ambiente Python e Imports
import os
import sys
import datetime 
import time
import pandas as pd # Import pandas as pd

# Ajusta o sys.path para que o Python encontre o diretório 'src'
# Assumindo que o notebook está em <project_root>/notebooks/
# e que o 'src' está em <project_root>/src/
current_notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_notebook_dir, os.pardir))
src_dir = os.path.join(project_root, 'src')

if src_dir not in sys.path:
    sys.path.append(src_dir)

# Importa o cliente da API Open-Meteo
# O caminho é 'django_app.__shared.open_meteo_client' conforme sua estrutura.
from django_app.__shared.open_meteo_client import OpenMeteoClient 


# Célula 3: Inicializar Cliente e Obter Dados para Coordenadas Fixas
try:
    # --- COORDENADAS FIXAS PARA TESTE (Fortaleza, Ceará, Brazil) ---
    test_latitude = -3.7319  
    test_longitude = -38.5267 
    test_location_name = "Fortaleza, Ceará, Brazil" 

    print(f"Testing with Fixed Location: '{test_location_name}' at ({test_latitude}, {test_longitude})")

    # Configura o cliente Open-Meteo
    client = OpenMeteoClient()

    # Define o período para buscar dados: DOIS DIAS ATRÁS
    # A API Open-Meteo Archive só fornece dados históricos (dias COMPLETA E FINALMENTE arquivados).
    # Pedir dados de 'hoje' ou 'ontem' (se o dia ainda não fechou em UTC) resulta em Nones.
    # Ex: Se hoje é 2025-06-15 10:00:00-03, UTC é 2025-06-15 13:00:00 UTC.
    # Pedimos dados de 2025-06-13, que já está arquivado.
    today_utc = datetime.datetime.now(datetime.timezone.utc).date()
    end_date = today_utc - datetime.timedelta(days=2) # Final do dia 2 dias atrás
    start_date = end_date # Para pegar apenas 1 dia de dados (24 horas)

    # Converte para objetos datetime "aware" em UTC para a API
    start_datetime_utc = datetime.datetime.combine(start_date, datetime.datetime.min.time()).replace(tzinfo=datetime.timezone.utc)
    end_datetime_utc = datetime.datetime.combine(end_date, datetime.datetime.max.time()).replace(tzinfo=datetime.timezone.utc)

    print(f"Fetching hourly data from {start_datetime_utc.strftime('%Y-%m-%d')} to {end_datetime_utc.strftime('%Y-%m-%d')} for {test_location_name}...")
    
    # Faz a chamada à API
    hourly_data = client.get_hourly_historical_weather(
        test_latitude, test_longitude,
        start_datetime_utc,
        end_datetime_utc
    )

    # Célula 4: Analisar e Imprimir os Resultados
    if hourly_data:
        print(f"\nData obtained for {test_location_name}: {len(hourly_data)} records")
        
        # Converte a lista de dicionários para um DataFrame do Pandas para melhor visualização.
        df_hourly_data = pd.DataFrame(hourly_data)
        
        # Exibe o DataFrame completo no notebook.
        print("\nDataFrame of Hourly Data (Last 24h):")
        print(df_hourly_data.to_string())

        # Opcional: Salva os dados em um arquivo CSV temporário para inspeção externa.
        # df_hourly_data.to_csv('hourly_test_data.csv', index=False)
        # print("\nData saved to hourly_test_data.csv for inspection.")

        # Verifica se algum campo essencial retornou como None.
        all_fields = ['recorded_at', 'temperature', 'humidity', 'wind_speed', 'wind_direction', 'pressure', 'rainfall']
        none_values_found = False
        for record in hourly_data:
            for field in all_fields:
                if record.get(field) is None:
                    print(f"ATTENTION: Field '{field}' is None in record from {record.get('recorded_at')}")
                    none_values_found = True
        if not none_values_found:
            print("\nAll required fields seem to have non-None values for this station.")
        else:
            print("\nWARNING: Some fields are None. Check the data in the DataFrame.")

    else:
        print("Could not obtain data for the test location. Check connectivity and API response.")

except Exception as e:
    print(f"Error in notebook: {e}")

Testing with Fixed Location: 'Fortaleza, Ceará, Brazil' at (-3.7319, -38.5267)
Fetching hourly data from 2025-06-14 to 2025-06-14 for Fortaleza, Ceará, Brazil...

Data obtained for Fortaleza, Ceará, Brazil: 24 records

DataFrame of Hourly Data (Last 24h):
                 recorded_at  temperature  humidity  wind_speed  wind_direction  pressure  rainfall
0  2025-06-14 03:00:00+00:00         27.0        80        4.42              96    1014.5       0.0
1  2025-06-14 04:00:00+00:00         25.7        84        4.13             107    1015.0       0.8
2  2025-06-14 05:00:00+00:00         25.5        81        5.35              98    1015.1       0.1
3  2025-06-14 06:00:00+00:00         26.1        77        4.46             106    1015.3       0.1
4  2025-06-14 07:00:00+00:00         26.8        69        6.09             118    1014.6       0.0
5  2025-06-14 08:00:00+00:00         26.3        72        5.83             123    1014.3       0.0
6  2025-06-14 09:00:00+00:00         26.1   